# Session 7 — Classification bake-off

I'm reopening Session 7 by reloading the model-ready features and re-splitting them with the same crystallized `ModelTrainer` I sealed in Session 6. Before training a single classifier, I print the persistence bar — the naive "tomorrow's category = today's category" scores — so every model this session has an honest number to beat. The Severe recall is the one I'm really hunting: the fraction of true-Severe days the naive forecast actually catches.

In [1]:
# --- Bucket 0: re-establish the contract and print the bar we must beat ---

%load_ext autoreload
%autoreload 2

import pandas as pd
from sklearn.metrics import recall_score

from src import config
from src.models.baseline import PersistenceBaseline
from src.models.trainer import ModelTrainer

features = pd.read_parquet(config.FEATURES_PATH)

trainer = ModelTrainer()
train, test = trainer.split(features)

baseline = PersistenceBaseline()
bar = baseline.classification_scores(test)

# Severe recall isn't in the baseline's summary dict, so compute it directly:
# of the truly-Severe test days, what fraction did "today's category" catch?
severe_recall = recall_score(
    test[config.TARGET_BUCKET_COL],
    test[config.AQI_BUCKET_COL],
    labels=["Severe"],
    average="macro",
    zero_division=0,
)

print(f"train rows: {len(train):,}   test rows: {len(test):,}")
print(f"baseline weighted-F1:   {bar['weighted_f1']:.3f}")
print(f"baseline macro-F1:      {bar['macro_f1']:.3f}")
print(f"baseline Severe recall: {severe_recall:.3f}")

train rows: 155,131   test rows: 46,594
baseline weighted-F1:   0.711
baseline macro-F1:      0.606
baseline Severe recall: 0.343


### Bucket 1 — Feature matrix, encoded target, and the class imbalance

I carve `X` (the model's feature contract from config) and the target `y` for
both train and test. I encode `target_bucket` into severity-ordered integers
(Good=0 → Severe=5) because XGBoost's classifier only accepts integer labels,
and I want one consistent `y` feeding every model. Then I print the training
class distribution in severity order — I expect Severe to be a tiny sliver,
which is exactly why macro-F1 and Severe recall are the metrics I trust here.

In [2]:
# --- Bucket 1: build X / y for classification and inspect the imbalance ---

# Severity-ordered label map, driven by the single source of truth in config.
label_to_id = {cat: i for i, cat in enumerate(config.AQI_CATEGORIES)}
id_to_label = {i: cat for cat, i in label_to_id.items()}

# X = the model's feature contract; y = the encoded next-day category.
X_train = train[config.MODEL_FEATURES]
X_test = test[config.MODEL_FEATURES]
y_train = train[config.TARGET_BUCKET_COL].map(label_to_id)
y_test = test[config.TARGET_BUCKET_COL].map(label_to_id)

# Fail loudly if any category slipped through the map unencoded.
assert y_train.isna().sum() == 0, "unmapped category in train target"
assert y_test.isna().sum() == 0, "unmapped category in test target"

print(f"X_train: {X_train.shape}   X_test: {X_test.shape}")
print(f"y_train: {len(y_train):,}   y_test: {len(y_test):,}")
print(f"label map: {label_to_id}\n")

# Training class distribution, in severity order (not count order).
dist = train[config.TARGET_BUCKET_COL].value_counts().reindex(config.AQI_CATEGORIES)
dist_pct = (dist / dist.sum() * 100).round(2)
print(pd.DataFrame({"count": dist, "pct": dist_pct}))

X_train: (155131, 22)   X_test: (46594, 22)
y_train: 155,131   y_test: 46,594
label map: {'Good': 0, 'Satisfactory': 1, 'Moderate': 2, 'Poor': 3, 'Very Poor': 4, 'Severe': 5}

               count    pct
target_bucket              
Good           19944  12.86
Satisfactory   52266  33.69
Moderate       55201  35.58
Poor           13219   8.52
Very Poor      12213   7.87
Severe          2288   1.47


### Bucket 2 — Lock the classification roster

I lock 7 classifiers across four families (linear, probabilistic, tree,
boosting) as `classification_roster()`. The SVM family is represented by
`LinearSVC`, not an RBF `SVC` — a kernel SVM is O(n^2)+ and won't finish on
155k rows across time-folds (same reason SVR was cut from regression). This
roster is deliberately UNBALANCED; Bucket 5 adds the balanced variants so I
can measure the Severe-recall lift honestly.

In [3]:
# --- Bucket 2: lock the 7-model classification roster (unbalanced baseline) ---
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


def classification_roster():
    """Return the 7-model classification roster for the bake-off.

    Seven estimators across four families — linear, probabilistic, tree, and
    boosting — so the bake-off compares fundamentally different ways of drawing
    class boundaries. Stochastic models (trees, boosting) take a random_state;
    LogisticRegression and GaussianNB are deterministic. The SVM family is
    represented by LinearSVC, not an RBF SVC: a kernel SVC is O(n^2)+ and won't
    finish on 155k rows across TimeSeriesSplit folds (same reason SVR was cut
    from regression). This roster is UNBALANCED on purpose — Bucket 5 adds the
    class_weight='balanced' variants so the Severe-recall lift is measurable.

    Returns:
        (name, estimator) pairs, each ready to wrap in the pipeline.
    """
    return [
        ("LogisticRegression", LogisticRegression(max_iter=1000)),
        ("GaussianNB",         GaussianNB()),
        ("LinearSVM",          LinearSVC(dual="auto", max_iter=2000)),
        ("DecisionTree",       DecisionTreeClassifier(random_state=config.RANDOM_STATE)),
        ("RandomForest",       RandomForestClassifier(random_state=config.RANDOM_STATE, n_jobs=-1)),
        ("XGBoost",            XGBClassifier(random_state=config.RANDOM_STATE, n_jobs=-1)),
        ("LightGBM",           LGBMClassifier(random_state=config.RANDOM_STATE, n_jobs=-1, verbose=-1)),
    ]


roster = classification_roster()
print(f"roster locked: {len(roster)} models\n")
for name, model in roster:
    print(f"  {name:<20} {type(model).__name__}")

roster locked: 7 models

  LogisticRegression   LogisticRegression
  GaussianNB           GaussianNB
  LinearSVM            LinearSVC
  DecisionTree         DecisionTreeClassifier
  RandomForest         RandomForestClassifier
  XGBoost              XGBClassifier
  LightGBM             LGBMClassifier


### Bucket 3 — A custom Severe-recall scorer

sklearn has no built-in string for "recall of the Severe class only," so I build
one with `make_scorer`. Recall = of all truly-Severe days, the fraction the model
actually flags — the metric that matters most for public health. I bundle it with
`f1_weighted` and `f1_macro` into one scoring dict, then sanity-check the scorer by
feeding it the persistence baseline's predictions: it must reproduce the 0.343
Severe recall I measured back in Bucket 0.

In [4]:
# --- Bucket 3: custom Severe-recall scorer + classification scoring dict ---
from sklearn.metrics import make_scorer, recall_score

# Severe is the last, rarest, most dangerous class. Its encoded id, from config.
SEVERE_ID = config.AQI_CATEGORIES.index("Severe")   # -> 5


def severe_recall_score(y_true, y_pred):
    """Recall for the Severe class only: of truly-Severe days, the fraction caught.

    Args:
        y_true: Integer-encoded true next-day categories.
        y_pred: Integer-encoded predicted categories.

    Returns:
        Recall for the single Severe class (0.0-1.0). zero_division=0 makes a
        fold with no Severe days score 0 rather than raise.
    """
    return recall_score(
        y_true, y_pred, labels=[SEVERE_ID], average="macro", zero_division=0
    )


# Wrap the metric so cross_validate can call it per fold (greater = better).
severe_recall_scorer = make_scorer(severe_recall_score)

# The scoring dict run_bakeoff will iterate. weighted_f1 is first -> it sorts the
# leaderboard; the other two ride along as columns. None are error metrics.
CLASSIFICATION_SCORING = {
    "weighted_f1":   "f1_weighted",
    "macro_f1":      "f1_macro",
    "severe_recall": severe_recall_scorer,
}

# --- Sanity check: the scorer must reproduce Bucket 0's baseline number ---
y_true_base = test[config.TARGET_BUCKET_COL].map(label_to_id)
y_pred_base = test[config.AQI_BUCKET_COL].map(label_to_id)
check = severe_recall_score(y_true_base, y_pred_base)

print(f"SEVERE_ID: {SEVERE_ID}")
print(f"scoring keys: {list(CLASSIFICATION_SCORING)}")
print(f"scorer on baseline preds -> Severe recall: {check:.3f}   (expect 0.343)")

SEVERE_ID: 5
scoring keys: ['weighted_f1', 'macro_f1', 'severe_recall']
scorer on baseline preds -> Severe recall: 0.343   (expect 0.343)


### Bucket 4 — Unbalanced classification bake-off (CV on training data only)

I cross-validate all 7 models with the sealed 5-fold TimeSeriesSplit, ranking on
weighted-F1 but watching `severe_recall` as the metric that really matters. The
test set stays untouched — this is pure model selection on the training frame.
This roster is deliberately UNBALANCED: it's the "before" photo, so Bucket 5's
class_weight='balanced' re-run has a starting line to prove its Severe-recall
lift against. I prototype a hardened copy of run_bakeoff here (it must accept a
callable scorer, not just strings); the one-line fix crystallizes at Bucket 8.

In [5]:
# --- Bucket 4: cross-validated bake-off, UNBALANCED roster ---
from sklearn.model_selection import cross_validate


def run_clf_bakeoff(trainer, roster, X, y, scoring):
    """Notebook prototype of the classification bake-off.

    A faithful copy of ModelTrainer.run_bakeoff with ONE hardening: the
    error-metric guard now checks isinstance(scorer, str) before calling
    .startswith, so a callable scorer (our severe_recall_scorer) no longer
    crashes the loop. This one-line fix folds back into the sealed run_bakeoff
    at Bucket 8 so a single method serves regression and classification.
    """
    cv = trainer.get_cv()
    rows = []
    for name, model in roster:
        pipe = trainer.make_pipeline(model)
        result = cross_validate(
            pipe, X, y,
            cv=cv,
            scoring=scoring,
            n_jobs=1,             # models parallelize internally; avoid nesting
            error_score="raise",  # fail loudly, never silently score NaN
        )
        row = {"model": name}
        for metric, scorer in scoring.items():
            mean_score = result[f"test_{metric}"].mean()
            # harden: only STRING scorers can be negated error metrics
            if isinstance(scorer, str) and scorer.startswith("neg_"):
                mean_score = -mean_score
            row[f"cv_{metric}"] = mean_score
        rows.append(row)

    first_metric = next(iter(scoring))                      # weighted_f1 -> sorts
    first_scorer = scoring[first_metric]
    best_is_low = isinstance(first_scorer, str) and first_scorer.startswith("neg_")
    return (
        pd.DataFrame(rows)
        .sort_values(f"cv_{first_metric}", ascending=best_is_low)
        .reset_index(drop=True)
    )


leaderboard = run_clf_bakeoff(
    trainer, roster, X_train, y_train, CLASSIFICATION_SCORING
)

print("UNBALANCED classification bake-off — 5-fold TimeSeriesSplit CV\n")
print(leaderboard.to_string(index=False))
print("\nbaseline to beat ->  weighted_f1: 0.711   severe_recall: 0.343")

UNBALANCED classification bake-off — 5-fold TimeSeriesSplit CV

             model  cv_weighted_f1  cv_macro_f1  cv_severe_recall
          LightGBM        0.691830     0.582602          0.318359
      RandomForest        0.689160     0.580610          0.300433
           XGBoost        0.686455     0.574709          0.294014
LogisticRegression        0.667974     0.543510          0.243867
      DecisionTree        0.570914     0.471973          0.324140
        GaussianNB        0.551160     0.490586          0.468814
         LinearSVM        0.547126     0.380808          0.061435

baseline to beat ->  weighted_f1: 0.711   severe_recall: 0.343


### Bucket 5 — Balanced re-run: the Severe-recall vs weighted-F1 trade

I re-run the bake-off with `class_weight="balanced"` on the 5 models that support
it (LogReg, LinearSVM, DecisionTree, RandomForest, LightGBM). GaussianNB and
XGBoost have no `class_weight` knob, so they ride unchanged and are flagged. I
then merge the balanced and unbalanced boards to read the trade directly: how
much Severe recall each model gained, and how much weighted-F1 it cost. I expect
Severe recall to jump well past the 0.343 baseline, with weighted-F1 falling.

In [6]:
# --- Bucket 5: balanced roster + before/after comparison ---

def balanced_classification_roster():
    """Return the roster with class_weight='balanced' where the model supports it.

    Balancing up-weights rare classes inversely to their frequency, so a missed
    Severe day is penalised heavily. Five sklearn/LightGBM models take the knob
    directly. GaussianNB (generative — no class_weight) and XGBoost (balances via
    per-row sample_weight, which doesn't compose with the shared CV loop) ride
    unchanged and are flagged; LightGBM keeps the boosting family balanced.

    Returns:
        (name, estimator) pairs, ready to wrap in the pipeline.
    """
    return [
        ("LogisticRegression", LogisticRegression(max_iter=1000, class_weight="balanced")),
        ("GaussianNB",         GaussianNB()),  # no class_weight; already recall-oriented
        ("LinearSVM",          LinearSVC(dual="auto", max_iter=2000, class_weight="balanced")),
        ("DecisionTree",       DecisionTreeClassifier(random_state=config.RANDOM_STATE, class_weight="balanced")),
        ("RandomForest",       RandomForestClassifier(random_state=config.RANDOM_STATE, n_jobs=-1, class_weight="balanced")),
        ("XGBoost",            XGBClassifier(random_state=config.RANDOM_STATE, n_jobs=-1)),  # no class_weight knob
        ("LightGBM",           LGBMClassifier(random_state=config.RANDOM_STATE, n_jobs=-1, verbose=-1, class_weight="balanced")),
    ]


balanced_roster = balanced_classification_roster()
leaderboard_bal = run_clf_bakeoff(
    trainer, balanced_roster, X_train, y_train, CLASSIFICATION_SCORING
)

print("BALANCED classification bake-off — 5-fold TimeSeriesSplit CV\n")
print(leaderboard_bal.to_string(index=False))

# --- The money shot: what balancing bought (recall) vs cost (F1) ---
compare = leaderboard[["model", "cv_weighted_f1", "cv_severe_recall"]].merge(
    leaderboard_bal[["model", "cv_weighted_f1", "cv_severe_recall"]],
    on="model", suffixes=("_unbal", "_bal"),
)
compare["severe_recall_gain"] = compare["cv_severe_recall_bal"] - compare["cv_severe_recall_unbal"]
compare["weighted_f1_change"] = compare["cv_weighted_f1_bal"] - compare["cv_weighted_f1_unbal"]
compare = compare.sort_values("cv_severe_recall_bal", ascending=False).reset_index(drop=True)

print("\n\nBEFORE -> AFTER balancing (sorted by balanced Severe recall)\n")
print(compare.round(3).to_string(index=False))
print("\nbaseline Severe recall to beat -> 0.343")

BALANCED classification bake-off — 5-fold TimeSeriesSplit CV

             model  cv_weighted_f1  cv_macro_f1  cv_severe_recall
           XGBoost        0.686455     0.574709          0.294014
      RandomForest        0.680712     0.563154          0.260968
          LightGBM        0.671126     0.593358          0.521271
LogisticRegression        0.636816     0.558568          0.662575
         LinearSVM        0.604851     0.509084          0.363866
      DecisionTree        0.567020     0.463833          0.299820
        GaussianNB        0.551160     0.490586          0.468814


BEFORE -> AFTER balancing (sorted by balanced Severe recall)

             model  cv_weighted_f1_unbal  cv_severe_recall_unbal  cv_weighted_f1_bal  cv_severe_recall_bal  severe_recall_gain  weighted_f1_change
LogisticRegression                 0.668                   0.244               0.637                 0.663               0.419              -0.031
          LightGBM                 0.692            

### Bucket 6 — Sealed test-set evaluation of the 2 finalists

I take the two CV finalists — LightGBM (balanced, best all-rounder) and
LogisticRegression (balanced, highest Severe recall) — fit each on the FULL
training set, and predict the held-out test set exactly once. I score both plus
the persistence baseline on the same test frame, so the comparison is finally
apples-to-apples. Then I read each confusion matrix in severity order, focusing
on the Severe row: how many true-Severe days each model catches, and how many it
dangerously mislabels as a 'safe' category.

In [7]:
# --- Bucket 6: fit finalists on full train, judge once on the sealed test set ---
from sklearn.metrics import f1_score, confusion_matrix

finalists = [
    ("LightGBM_balanced", LGBMClassifier(
        random_state=config.RANDOM_STATE, n_jobs=-1, verbose=-1, class_weight="balanced")),
    ("LogReg_balanced", LogisticRegression(
        max_iter=1000, class_weight="balanced")),
]

results = []
predictions = {}
for name, model in finalists:
    pipe = trainer.make_pipeline(model)
    pipe.fit(X_train, y_train)        # fit on ALL training data
    y_pred = pipe.predict(X_test)     # predict the sealed test set — one look
    predictions[name] = y_pred
    results.append({
        "model": name,
        "weighted_f1":  f1_score(y_test, y_pred, average="weighted", zero_division=0),
        "macro_f1":     f1_score(y_test, y_pred, average="macro", zero_division=0),
        "severe_recall": severe_recall_score(y_test, y_pred),
    })

# Baseline on the SAME test frame -> finally a clean head-to-head.
y_pred_base = test[config.AQI_BUCKET_COL].map(label_to_id)
results.append({
    "model": "PersistenceBaseline",
    "weighted_f1":  f1_score(y_test, y_pred_base, average="weighted", zero_division=0),
    "macro_f1":     f1_score(y_test, y_pred_base, average="macro", zero_division=0),
    "severe_recall": severe_recall_score(y_test, y_pred_base),
})

board = pd.DataFrame(results)
print("SEALED TEST-SET evaluation — finalists vs baseline\n")
print(board.round(3).to_string(index=False))

# --- Confusion matrices: hunt the dangerous lower-left corner ---
severity = config.AQI_CATEGORIES
label_ids = list(range(len(severity)))          # 0..5, severity order

for name, y_pred in predictions.items():
    cm = confusion_matrix(y_test, y_pred, labels=label_ids)
    cm_df = pd.DataFrame(cm, index=severity, columns=severity)
    cm_df.index.name = "actual"
    cm_df.columns.name = "predicted"
    print(f"\n=== {name} ===")
    print(cm_df.to_string())

    severe_row = cm[SEVERE_ID]                   # actual == Severe
    total = severe_row.sum()
    caught = severe_row[SEVERE_ID]
    called_safe = severe_row[[0, 1, 2]].sum()    # Good / Satisfactory / Moderate
    print(f"true Severe days: {total} | caught: {caught} "
          f"| missed: {total - caught} | called 'safe': {called_safe}")

SEALED TEST-SET evaluation — finalists vs baseline

              model  weighted_f1  macro_f1  severe_recall
  LightGBM_balanced        0.685     0.586          0.535
    LogReg_balanced        0.654     0.551          0.606
PersistenceBaseline        0.711     0.606          0.343

=== LightGBM_balanced ===
predicted     Good  Satisfactory  Moderate  Poor  Very Poor  Severe
actual                                                             
Good          6536          1076        75    11          4      11
Satisfactory  2789         11138      2108   209         28      16
Moderate       154          2304      9728  3172        306      61
Poor             8            43       564  2454        696      88
Very Poor       10            21        85   782       1621     283
Severe           5             4         6    29         55     114
true Severe days: 213 | caught: 114 | missed: 99 | called 'safe': 15

=== LogReg_balanced ===
predicted     Good  Satisfactory  Moderate  Poor  V

### Bucket 7 — Tune the two finalists (time-aware, honest)

I tune both finalists with the sealed 5-fold TimeSeriesSplit. GridSearchCV sweeps
LogisticRegression's `C`; RandomizedSearchCV samples LightGBM's tree knobs. Both
optimise `macro_f1` (refit) so improvement means better balance across ALL six
classes, not just recall gaming. I compare each tuned model's CV scores against
its untuned balanced baseline — if the lift is negligible, I crystallize the
simple defaults and document tuning as an honest wash, exactly like regression.

In [8]:
# --- Bucket 7: tune the two finalists on TimeSeriesSplit CV ---
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

cv = trainer.get_cv()

# --- LogReg: sweep the regularization strength C ---
logreg_pipe = trainer.make_pipeline(
    LogisticRegression(max_iter=1000, class_weight="balanced"))
logreg_search = GridSearchCV(
    logreg_pipe,
    {"model__C": [0.01, 0.1, 1.0, 10.0]},
    cv=cv, scoring=CLASSIFICATION_SCORING, refit="macro_f1", n_jobs=1,
)
logreg_search.fit(X_train, y_train)

# --- LightGBM: sample its tree-shape knobs ---
lgbm_pipe = trainer.make_pipeline(LGBMClassifier(
    random_state=config.RANDOM_STATE, n_jobs=-1, verbose=-1, class_weight="balanced"))
lgbm_search = RandomizedSearchCV(
    lgbm_pipe,
    {
        "model__n_estimators":      [100, 200, 300, 500],
        "model__num_leaves":        [15, 31, 63],
        "model__learning_rate":     [0.01, 0.05, 0.1],
        "model__min_child_samples": [20, 50, 100],
    },
    n_iter=10, cv=cv, scoring=CLASSIFICATION_SCORING, refit="macro_f1",
    random_state=config.RANDOM_STATE, n_jobs=1,
)
lgbm_search.fit(X_train, y_train)


def best_cv_scores(search):
    """Pull all three CV metrics at the search's best (refit) setting."""
    i = search.best_index_
    r = search.cv_results_
    return {
        "weighted_f1":   r["mean_test_weighted_f1"][i],
        "macro_f1":      r["mean_test_macro_f1"][i],
        "severe_recall": r["mean_test_severe_recall"][i],
    }


# Untuned balanced CV scores from Bucket 5 (the honest "before").
untuned = {
    "LogReg_balanced":   {"weighted_f1": 0.637, "macro_f1": 0.559, "severe_recall": 0.663},
    "LightGBM_balanced": {"weighted_f1": 0.671, "macro_f1": 0.593, "severe_recall": 0.521},
}

for name, search in [("LogReg_balanced", logreg_search),
                     ("LightGBM_balanced", lgbm_search)]:
    tuned = best_cv_scores(search)
    print(f"=== {name} ===")
    print(f"  best params : {search.best_params_}")
    print(f"  untuned CV  : {untuned[name]}")
    print(f"  tuned   CV  : { {k: round(v, 3) for k, v in tuned.items()} }")
    print(f"  macro_f1 lift: {tuned['macro_f1'] - untuned[name]['macro_f1']:+.3f}\n")

=== LogReg_balanced ===
  best params : {'model__C': 10.0}
  untuned CV  : {'weighted_f1': 0.637, 'macro_f1': 0.559, 'severe_recall': 0.663}
  tuned   CV  : {'weighted_f1': np.float64(0.638), 'macro_f1': np.float64(0.559), 'severe_recall': np.float64(0.661)}
  macro_f1 lift: +0.000

=== LightGBM_balanced ===
  best params : {'model__num_leaves': 63, 'model__n_estimators': 500, 'model__min_child_samples': 20, 'model__learning_rate': 0.01}
  untuned CV  : {'weighted_f1': 0.671, 'macro_f1': 0.593, 'severe_recall': 0.521}
  tuned   CV  : {'weighted_f1': np.float64(0.674), 'macro_f1': np.float64(0.597), 'severe_recall': np.float64(0.528)}
  macro_f1 lift: +0.004



In [9]:
# --- Bucket 8 verify: crystallized trainer reproduces the notebook numbers ---
import importlib
from src import config
from src.models import trainer as trainer_mod
importlib.reload(trainer_mod)                 # pick up the rewritten file
from src.models.trainer import ModelTrainer

t = ModelTrainer()

# 1) config imported the scorer without a circular-import error
print("scoring keys:", list(config.CLASSIFICATION_SCORING))

# 2) balanced flag flips class_weight exactly where it should
unbal = dict(t.classification_roster(balanced=False))
bal   = dict(t.classification_roster(balanced=True))
print("LogReg unbalanced weight:", unbal["LogisticRegression"].class_weight)
print("LogReg balanced   weight:", bal["LogisticRegression"].class_weight)
print("GaussianNB has class_weight attr:", hasattr(bal["GaussianNB"], "class_weight"))

# 3) re-run the balanced bake-off from the SEALED module — must match Bucket 5
board = t.run_bakeoff(
    t.classification_roster(balanced=True),
    X_train, y_train, config.CLASSIFICATION_SCORING,
)
print("\nsealed-module balanced bake-off:")
print(board.round(3).to_string(index=False))

scoring keys: ['weighted_f1', 'macro_f1', 'severe_recall']
LogReg unbalanced weight: None
LogReg balanced   weight: balanced
GaussianNB has class_weight attr: False

sealed-module balanced bake-off:
             model  cv_weighted_f1  cv_macro_f1  cv_severe_recall
           XGBoost           0.686        0.575             0.294
      RandomForest           0.681        0.563             0.261
          LightGBM           0.671        0.593             0.521
LogisticRegression           0.637        0.559             0.663
         LinearSVM           0.605        0.509             0.364
      DecisionTree           0.567        0.464             0.300
        GaussianNB           0.551        0.491             0.469
